In [1]:
# Imports and paths
import os
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

In [7]:
BASE = ""
OUT = "outputs/dashboard_data"
os.makedirs(OUT, exist_ok=True)

In [8]:
df = pd.read_csv("dataset/marketing_AB_clean.csv")

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
day_num = {d: i + 1 for i, d in enumerate(day_order)}
df["day_num"] = df["most_ads_day"].map(day_num)

In [9]:
bins = [0, 10, 25, 50, 100, df["total_ads"].max()]
labels = ["1-10", "11-25", "26-50", "51-100", "100+"]
df["ads_bucket"] = pd.cut(df["total_ads"], bins=bins, labels=labels, include_lowest=True)
bucket_num = {b: i + 1 for i, b in enumerate(labels)}

In [10]:
# Helper: users, conversions and rate per group within a dimension
def rate_table(data, dims):
    out = (
        data.groupby(dims + ["test_group"], observed=True)["converted_int"]
        .agg(users="count", conversions="sum")
        .reset_index()
    )
    out["conversion_rate"] = out["conversions"] / out["users"]
    return out

In [11]:
# 1. KPI cards: one row per group
kpi_groups = rate_table(df, [])
kpi_groups.to_csv(f"{OUT}/kpi_groups.csv", index=False)
print(kpi_groups)

  test_group   users  conversions  conversion_rate
0         ad  564577        14423         0.025547
1        psa   23524          420         0.017854


In [12]:
# 2. KPI cards: overall test result (one row)
ad = df[df["test_group"] == "ad"]["converted_int"]
psa = df[df["test_group"] == "psa"]["converted_int"]
z, p = proportions_ztest([ad.sum(), psa.sum()], [len(ad), len(psa)], alternative="larger")
low, high = confint_proportions_2indep(ad.sum(), len(ad), psa.sum(), len(psa),
                                       method="wald", compare="diff", alpha=0.05)
abs_lift = ad.mean() - psa.mean()

kpi_test = pd.DataFrame([{
    "ad_rate": ad.mean(),
    "psa_rate": psa.mean(),
    "abs_lift_pp": abs_lift * 100,
    "rel_lift_pct": abs_lift / psa.mean() * 100,
    "ci_low_pp": low * 100,
    "ci_high_pp": high * 100,
    "z_stat": z,
    "p_value": p,
    "significant_5pct": p < 0.05,
}])
kpi_test.to_csv(f"{OUT}/kpi_test.csv", index=False)
print(kpi_test.T)

                          0
ad_rate            0.025547
psa_rate           0.017854
abs_lift_pp        0.769245
rel_lift_pct      43.085064
ci_low_pp          0.595093
ci_high_pp         0.943397
z_stat             7.370078
p_value                 0.0
significant_5pct       True


In [13]:
# 3. Conversion by day / hour / ad-exposure bucket (with sort-order columns)
by_day = rate_table(df, ["most_ads_day", "day_num"])
by_day.to_csv(f"{OUT}/by_day.csv", index=False)

In [ ]:


# %% 4. Day x hour heatmap table (long format for a matrix visual)
heat = rate_table(df, ["most_ads_day", "day_num", "most_ads_hour"])
heat.to_csv(f"{OUT}/heatmap_day_hour.csv", index=False)


# %% 5. Segment lift table (ad vs psa inside each segment), with Bonferroni flag
def segment_lift(data, col, seg_type):
    rows = []
    for seg, g in data.groupby(col, observed=True):
        a = g[g["test_group"] == "ad"]["converted_int"]
        s = g[g["test_group"] == "psa"]["converted_int"]
        if len(a) < 30 or len(s) < 30:
            continue
        _, pv = proportions_ztest([a.sum(), s.sum()], [len(a), len(s)], alternative="larger")
        rows.append({
            "segment_type": seg_type,
            "segment": str(seg),
            "ad_rate": a.mean(),
            "psa_rate": s.mean(),
            "abs_lift_pp": (a.mean() - s.mean()) * 100,
            "p_value": pv,
        })
    return pd.DataFrame(rows)


seg = pd.concat([
    segment_lift(df, "most_ads_day", "Day"),
    segment_lift(df, "most_ads_hour", "Hour"),
    segment_lift(df, "ads_bucket", "Ad exposure"),
], ignore_index=True)
seg["significant_bonferroni"] = seg["p_value"] < 0.05 / len(seg)
seg.to_csv(f"{OUT}/segment_lift.csv", index=False)
print(seg.round(4))

# %% 6. Business impact (placeholder assumptions: CHANGE and state in README)
value_per_conversion = 50
cost_per_ad_user = 0.50

n_ad = len(ad)
extra_conv = abs_lift * n_ad
impact = pd.DataFrame([{
    "users_exposed": n_ad,
    "extra_conversions": extra_conv,
    "value_per_conversion": value_per_conversion,
    "incremental_revenue": extra_conv * value_per_conversion,
    "cost_per_user": cost_per_ad_user,
    "total_cost": cost_per_ad_user * n_ad,
}])
impact["roi_pct"] = (impact["incremental_revenue"] - impact["total_cost"]) / impact["total_cost"] * 100
impact.to_csv(f"{OUT}/business_impact.csv", index=False)
print(impact.T)

# %% 7. Check exported files
for f in sorted(os.listdir(OUT)):
    print(f, pd.read_csv(f"{OUT}/{f}").shape)